# KKBox Music Streaming Cohort & Retention Analysis — Data Preparation

This notebook is the not-glamorous-but-necessary half of the KKBox project.

The listening logs contain more than 400 million daily user records, which is far too much to casually load into memory and poke around with. The goal here is to turn that raw history into something the cohort analysis can actually use: **one row per user per activity month**.

Most of the work is therefore about checking coverage, processing the logs in chunks, making sure chunk boundaries don't double-count anything, and saving a clean monthly history for the analysis notebook.

> **Data note:** The original KKBox competition files stay local and are not included in the public repository.


## A note before rerunning this

A few parts of this notebook are expensive.

The listening logs are read in 1,000,000-row chunks and the partial aggregates are written to Parquet, so `pyarrow` or `fastparquet` is required. The historical log alone contains hundreds of millions of rows and takes a long time to process.

I've kept the executed outputs in the notebook so the pipeline can be reviewed without rerunning the slowest cells. The temporary chunk files are just intermediate artifacts; the next notebook uses the consolidated monthly files in `data/processed/user_month_history/`.


In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw")

print("KKBox data directory:", DATA_DIR.resolve())

KKBox data directory: /Users/dorisxia/Downloads/data-portfolio/kkbox-retention-analysis/data/raw


## 1. What time period do the raw files actually cover?

Before combining anything, I want to know whether the original and `v2` files overlap, replace each other, or cover different periods. With this dataset, that matters a lot more than the filenames make it sound.


In [2]:
def inspect_date_range(filepath, date_column, chunksize=1_000_000):
    min_date = None
    max_date = None
    rows = 0

    for chunk in pd.read_csv(
        filepath,
        usecols=[date_column],
        chunksize=chunksize
    ):
        dates = pd.to_numeric(chunk[date_column], errors="coerce").dropna()

        if len(dates) == 0:
            continue

        chunk_min = dates.min()
        chunk_max = dates.max()

        min_date = chunk_min if min_date is None else min(min_date, chunk_min)
        max_date = chunk_max if max_date is None else max(max_date, chunk_max)

        rows += len(chunk)

    return {
        "file": filepath.name,
        "rows": rows,
        "min_date": int(min_date) if min_date is not None else None,
        "max_date": int(max_date) if max_date is not None else None
    }

In [3]:
transaction_results = []

for filename in ["transactions.csv", "transactions_v2.csv"]:
    result = inspect_date_range(
        DATA_DIR / filename,
        "transaction_date"
    )
    transaction_results.append(result)

pd.DataFrame(transaction_results)

,file,rows,min_date,max_date
0,transactions.csv,21547746,20150101,20170228
1,transactions_v2.csv,1431009,20150101,20170331


## 2. A quick look at the churn labels

I'm not using the competition churn label as my main retention definition, but I still want to understand what the two training files contain and how their target distributions differ.


In [4]:
train = pd.read_csv(DATA_DIR / "train.csv")
train_v2 = pd.read_csv(DATA_DIR / "train_v2.csv")

print("train.csv")
print("Shape:", train.shape)
print(train.head())
print()
print("Columns:", train.columns.tolist())

print("\ntrain_v2.csv")
print("Shape:", train_v2.shape)
print(train_v2.head())
print()
print("Columns:", train_v2.columns.tolist())

train.csv
Shape: (992931, 2)
                                           msno  is_churn
0  waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=         1
1  QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=         1
2  fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=         1
3  mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=         1
4  XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=         1

Columns: ['msno', 'is_churn']

train_v2.csv
Shape: (970960, 2)
                                           msno  is_churn
0  ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=         1
1  f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=         1
2  zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=         1
3  8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=         1
4  K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=         1

Columns: ['msno', 'is_churn']


In [5]:
print("train.csv")
print(train["is_churn"].value_counts())
print()
print(train["is_churn"].value_counts(normalize=True))

print("\ntrain_v2.csv")
print(train_v2["is_churn"].value_counts())
print()
print(train_v2["is_churn"].value_counts(normalize=True))

train.csv
is_churn
0    929460
1     63471
Name: count, dtype: int64

is_churn
0    0.936077
1    0.063923
Name: proportion, dtype: float64

train_v2.csv
is_churn
0    883630
1     87330
Name: count, dtype: int64

is_churn
0    0.910058
1    0.089942
Name: proportion, dtype: float64


## 3. Checking listening-log coverage

These are the files that matter most for the cohort analysis. They're too large to load all at once, so I'm only scanning the date column in chunks to establish the observation window.


In [6]:
log_results = []

for filename in ["user_logs.csv", "user_logs_v2.csv"]:
    result = inspect_date_range(
        DATA_DIR / filename,
        "date"
    )
    log_results.append(result)

pd.DataFrame(log_results)

,file,rows,min_date,max_date
0,user_logs.csv,392106543,20150101,20170228
1,user_logs_v2.csv,18396362,20170301,20170331


## 4. Member data and acquisition cohorts

Registration date will define when each user's cohort starts.

I'm restricting acquisition cohorts to registrations that fall inside the listening-log window. Otherwise I'd be assigning retention outcomes to users during periods when I couldn't actually observe their listening behavior.


In [7]:
members = pd.read_csv(DATA_DIR / "members_v3.csv")

print("Shape:", members.shape)
print()
print(members.info())
print()
print(members.head())

Shape: (6769473, 6)

<class 'pandas.DataFrame'>
RangeIndex: 6769473 entries, 0 to 6769472
Data columns (total 6 columns):
 #   Column                  Dtype
---  ------                  -----
 0   msno                    str  
 1   city                    int64
 2   bd                      int64
 3   gender                  str  
 4   registered_via          int64
 5   registration_init_time  int64
dtypes: int64(4), str(2)
memory usage: 309.9 MB
None

                                           msno  city  bd  gender  \
0  Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=     1   0     NaN   
1  +tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=     1   0     NaN   
2  cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=     1   0     NaN   
3  9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=     1   0     NaN   
4  WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=     6  32  female   

   registered_via  registration_init_time  
0              11                20110911  
1               7                20110914

In [8]:
print("Unique users:", f"{members['msno'].nunique():,}")
print("Duplicate user IDs:", f"{members['msno'].duplicated().sum():,}")
print()

print("Missing values:")
print(members.isna().sum())

Unique users: 6,769,473
Duplicate user IDs: 0

Missing values:
msno                            0
city                            0
bd                              0
gender                    4429505
registered_via                  0
registration_init_time          0
dtype: int64


In [9]:
registration_dates = pd.to_datetime(
    members["registration_init_time"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

print("Earliest registration:", registration_dates.min())
print("Latest registration:", registration_dates.max())
print("Invalid registration dates:", registration_dates.isna().sum())

Earliest registration: 2004-03-26 00:00:00
Latest registration: 2017-04-29 00:00:00
Invalid registration dates: 0


In [10]:
members["registration_date"] = pd.to_datetime(
    members["registration_init_time"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

members["cohort_month"] = members["registration_date"].dt.to_period("M")

cohort_candidates = members[
    (members["registration_date"] >= "2015-01-01") &
    (members["registration_date"] <= "2017-03-31")
].copy()

print("Total members:", f"{len(members):,}")
print("Members registered during observable log period:",
      f"{len(cohort_candidates):,}")
print(
    "Share of members eligible for acquisition cohorts:",
    f"{len(cohort_candidates) / len(members):.1%}"
)

Total members: 6,769,473
Members registered during observable log period: 4,293,876
Share of members eligible for acquisition cohorts: 63.4%


In [11]:
cohort_sizes = (
    cohort_candidates
    .groupby("cohort_month", observed=True)["msno"]
    .nunique()
    .reset_index(name="registered_users")
)

print(cohort_sizes.to_string(index=False))

cohort_month  registered_users
     2015-01            102290
     2015-02             92790
     2015-03             95388
     2015-04             80934
     2015-05             71404
     2015-06             79167
     2015-07            111526
     2015-08            161470
     2015-09            119370
     2015-10            235884
     2015-11            245875
     2015-12            224427
     2016-01            253156
     2016-02            208170
     2016-03            195618
     2016-04            178329
     2016-05            184404
     2016-06            189359
     2016-07            189273
     2016-08            176584
     2016-09            163065
     2016-10            170302
     2016-11            165057
     2016-12            173444
     2017-01            163721
     2017-02            163797
     2017-03             99072


## 5. Test the daily-to-monthly transformation first

Before sending hundreds of millions of rows through anything, I'm testing the aggregation on a manageable sample.

The final grain I want is one row per user per month, with active days and listening totals rolled up from the daily logs.

One naming detail worth keeping straight: `daily_unique_songs_sum` is the **sum of the daily `num_unq` values**. It is not the number of distinct songs a user heard across the entire month.


In [4]:

log_sample = pd.read_csv(
    DATA_DIR / "user_logs.csv",
    nrows=1_000_000
)

log_sample["activity_date"] = pd.to_datetime(
    log_sample["date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

log_sample["activity_month"] = (
    log_sample["activity_date"].dt.to_period("M")
)

monthly_sample = (
    log_sample
    .groupby(["msno", "activity_month"], observed=True)
    .agg(
        active_days=("date", "nunique"),
        total_secs=("total_secs", "sum"),
        unique_songs=("num_unq", "sum"),
        songs_25=("num_25", "sum"),
        songs_50=("num_50", "sum"),
        songs_75=("num_75", "sum"),
        songs_985=("num_985", "sum"),
        songs_100=("num_100", "sum")
    )
    .reset_index()
)

print("Raw sample rows:", f"{len(log_sample):,}")
print("Aggregated user-month rows:", f"{len(monthly_sample):,}")
print()
monthly_sample.head(10)

Raw sample rows: 1,000,000
Aggregated user-month rows: 1,000,000



,msno,activity_month,active_days,total_secs,unique_songs,songs_25,songs_50,songs_75,songs_985,songs_100
0,+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=,2016-09,1,2998.267,10,1,0,1,1,10
1,+++dz9ZCWE2HB/47pJU82NJXQzQuZDx1Wm50YSk/kKk=,2016-03,1,17132.904,54,3,0,0,1,65
2,++5IYGT7+CWMJ8hRsqoQLoaTXBLMJzlfK12eMTr9Ilw=,2015-03,1,448.574,2,0,0,0,0,2
3,++7jYuHyUSp41PyuttFx/MCepv7TdFQULgN8TxZULZk=,2015-04,1,1693.113,10,4,0,0,1,6
4,++7jYuHyUSp41PyuttFx/MCepv7TdFQULgN8TxZULZk=,2015-08,1,2868.131,19,8,2,1,0,10
5,++7jYuHyUSp41PyuttFx/MCepv7TdFQULgN8TxZULZk=,2016-01,1,222.424,1,0,0,0,0,1
6,++95tJZADNg8U8HKbYdxbbXIRsO6pw1zBK4tHI7BtZo=,2015-01,1,1820.241,6,0,0,0,0,7
7,++95tJZADNg8U8HKbYdxbbXIRsO6pw1zBK4tHI7BtZo=,2015-03,1,6669.718,40,15,12,2,4,15
8,++95tJZADNg8U8HKbYdxbbXIRsO6pw1zBK4tHI7BtZo=,2015-06,1,1137.764,7,2,0,0,0,5
9,++95tJZADNg8U8HKbYdxbbXIRsO6pw1zBK4tHI7BtZo=,2015-07,1,8027.300,33,3,1,0,1,31


In [5]:
del log_sample
del monthly_sample

## 6. Build the chunked pipeline

Now for the part that makes the full dataset manageable.

Each million-row chunk gets reduced to partial user-month aggregates and written to disk. That means I never need the entire listening history sitting in memory at once.


In [6]:
from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path("../data/processed")
TEMP_DIR = PROCESSED_DIR / "monthly_chunks"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
def aggregate_log_file(filepath, source_name, chunksize=1_000_000):
    """
    Convert daily KKBox listening logs into partial user-month aggregates.

    Each raw chunk is aggregated independently and written to disk.
    A later consolidation step will combine user-months that appear
    across multiple chunks.
    """

    usecols = [
        "msno", "date",
        "num_25", "num_50", "num_75", "num_985",
        "num_100", "num_unq", "total_secs"
    ]

    for chunk_number, chunk in enumerate(
        pd.read_csv(filepath, usecols=usecols, chunksize=chunksize),
        start=1
    ):
        chunk["activity_date"] = pd.to_datetime(
            chunk["date"].astype(str),
            format="%Y%m%d",
            errors="coerce"
        )

        chunk = chunk.dropna(subset=["activity_date"])

        chunk["activity_month"] = (
            chunk["activity_date"].dt.to_period("M").astype(str)
        )

        monthly = (
            chunk
            .groupby(["msno", "activity_month"], observed=True)
            .agg(
                active_days=("date", "nunique"),
                total_secs=("total_secs", "sum"),
                daily_unique_songs_sum=("num_unq", "sum"),
                songs_25=("num_25", "sum"),
                songs_50=("num_50", "sum"),
                songs_75=("num_75", "sum"),
                songs_985=("num_985", "sum"),
                songs_100=("num_100", "sum")
            )
            .reset_index()
        )

        output_path = (
            TEMP_DIR /
            f"{source_name}_chunk_{chunk_number:04d}.parquet"
        )

        monthly.to_parquet(output_path, index=False)

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: processed "
                f"{chunk_number * chunksize:,} raw rows"
            )

    print(f"{source_name}: complete.")

### 6.1 Try it on March 2017 first

I'm using `user_logs_v2.csv` as the full-scale test before touching the much larger historical file. It only covers March 2017, so it's easier to validate the final user-month counts and make sure nobody somehow ends up with more than 31 active days.


In [8]:
aggregate_log_file(
    DATA_DIR / "user_logs_v2.csv",
    source_name="logs_v2"
)

logs_v2: processed 10,000,000 raw rows
logs_v2: complete.


In [9]:
chunk_files = sorted(TEMP_DIR.glob("logs_v2_chunk_*.parquet"))

print(f"Number of chunk files: {len(chunk_files)}")

for file in chunk_files[:5]:
    print(file.name)

Number of chunk files: 19
logs_v2_chunk_0001.parquet
logs_v2_chunk_0002.parquet
logs_v2_chunk_0003.parquet
logs_v2_chunk_0004.parquet
logs_v2_chunk_0005.parquet


In [10]:
total_rows = 0

for file in chunk_files:
    chunk_df = pd.read_parquet(file)
    total_rows += len(chunk_df)

print(f"Total partial user-month rows: {total_rows:,}")

Total partial user-month rows: 10,968,767


In [11]:
sample_chunk = pd.read_parquet(chunk_files[0])

print(sample_chunk.shape)
sample_chunk.head()

(591758, 10)


,msno,activity_month,active_days,total_secs,daily_unique_songs_sum,songs_25,songs_50,songs_75,songs_985,songs_100
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,2017-03,2,3413.584,19,3,2,1,0,13
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,2017-03,1,924.747,5,1,0,0,1,3
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,2017-03,1,1395.247,7,0,0,0,1,6
3,++0+IdHga8fCSioOVpU8K7y4Asw8AveIApVH2r9q9yY=,2017-03,1,10027.239,53,11,1,2,2,38
4,++0/NopttBsaAn6qHZA2AWWrDg7Me7UOMs1vsyo4tSI=,2017-03,2,1650.342,9,5,0,2,0,6


### 6.2 Chunk boundaries create duplicates — by design

A user's March activity can be split across several raw chunks, so the same user-month can appear in multiple temporary Parquet files.

That's expected. These are **partial** aggregates, not the final monthly table. I need one more consolidation pass to add those pieces back together.


In [12]:

keys = []

for file in chunk_files:
    part = pd.read_parquet(
        file,
        columns=["msno", "activity_month"]
    )
    keys.append(part)

all_keys = pd.concat(keys, ignore_index=True)

print("Partial user-month rows:", f"{len(all_keys):,}")
print(
    "Unique user-month combinations:",
    f"{all_keys.drop_duplicates().shape[0]:,}"
)
print(
    "Duplicate partial rows requiring consolidation:",
    f"{all_keys.duplicated(['msno', 'activity_month']).sum():,}"
)

Partial user-month rows: 10,968,767
Unique user-month combinations: 1,103,894
Duplicate partial rows requiring consolidation: 9,864,873


In [13]:
counts = (
    all_keys
    .groupby(["msno", "activity_month"])
    .size()
)

print(counts.describe())
print()
print("Maximum number of chunks containing one user-month:", counts.max())

count    1.103894e+06
mean     9.936431e+00
std      5.009027e+00
min      1.000000e+00
25%      6.000000e+00
50%      1.100000e+01
75%      1.400000e+01
max      1.900000e+01
dtype: float64

Maximum number of chunks containing one user-month: 19


In [14]:
del all_keys, keys, counts

### 6.3 Put March back together

Now I can combine the partial March rows into one record per user-month.

The sanity check here is simple but useful: after consolidation, there should be no duplicate user-months and nobody should have more than 31 active days.


In [15]:
march_parts = [
    pd.read_parquet(file)
    for file in chunk_files
]

march_partial = pd.concat(march_parts, ignore_index=True)

print("Partial rows:", f"{len(march_partial):,}")

Partial rows: 10,968,767


In [16]:

sum_cols = [
    "active_days",
    "total_secs",
    "daily_unique_songs_sum",
    "songs_25",
    "songs_50",
    "songs_75",
    "songs_985",
    "songs_100"
]

march_monthly = (
    march_partial
    .groupby(
        ["msno", "activity_month"],
        as_index=False,
        observed=True
    )[sum_cols]
    .sum()
)

print("Final March user-month rows:", f"{len(march_monthly):,}")
print(
    "Duplicate user-months:",
    march_monthly.duplicated(["msno", "activity_month"]).sum()
)

march_monthly.head()

Final March user-month rows: 1,103,894
Duplicate user-months: 0


,msno,activity_month,active_days,total_secs,daily_unique_songs_sum,songs_25,songs_50,songs_75,songs_985,songs_100
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,2017-03,26,117907.425,530,86,11,10,5,472
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,2017-03,31,192527.892,885,191,90,75,144,589
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,2017-03,28,115411.260,468,43,12,15,12,485
3,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,2017-03,21,149896.558,828,207,163,100,64,436
4,++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk=,2017-03,29,116433.247,230,105,24,39,35,479


In [17]:
print(march_monthly["active_days"].describe())

print(
    "Users with active_days > 31:",
    (march_monthly["active_days"] > 31).sum()
)

print(
    "Maximum active_days:",
    march_monthly["active_days"].max()
)

count    1.103894e+06
mean     1.666497e+01
std      1.030333e+01
min      1.000000e+00
25%      7.000000e+00
50%      1.800000e+01
75%      2.600000e+01
max      3.100000e+01
Name: active_days, dtype: float64
Users with active_days > 31: 0
Maximum active_days: 31


## 7. Run the full historical log

March worked, so now I'm applying the same pipeline to `user_logs.csv`, which covers January 2015 through February 2017.

This is the cell that takes forever. The output is saved to disk specifically so there is no reason to rerun it during normal analysis.


In [18]:
# same pipeline as March, just on the file that takes forever
aggregate_log_file(
    DATA_DIR / "user_logs.csv",
    source_name="logs"
)

logs: processed 10,000,000 raw rows
logs: processed 20,000,000 raw rows
logs: processed 30,000,000 raw rows
logs: processed 40,000,000 raw rows
logs: processed 50,000,000 raw rows
logs: processed 60,000,000 raw rows
logs: processed 70,000,000 raw rows
logs: processed 80,000,000 raw rows
logs: processed 90,000,000 raw rows
logs: processed 100,000,000 raw rows
logs: processed 110,000,000 raw rows
logs: processed 120,000,000 raw rows
logs: processed 130,000,000 raw rows
logs: processed 140,000,000 raw rows
logs: processed 150,000,000 raw rows
logs: processed 160,000,000 raw rows
logs: processed 170,000,000 raw rows
logs: processed 180,000,000 raw rows
logs: processed 190,000,000 raw rows
logs: processed 200,000,000 raw rows
logs: processed 210,000,000 raw rows
logs: processed 220,000,000 raw rows
logs: processed 230,000,000 raw rows
logs: processed 240,000,000 raw rows
logs: processed 250,000,000 raw rows
logs: processed 260,000,000 raw rows
logs: processed 270,000,000 raw rows
logs: proc

## 8. Consolidate the historical chunks month by month

There are hundreds of temporary Parquet files at this point. Rather than trying to concatenate all of them into one enormous DataFrame, I'm consolidating one calendar month at a time.

It's slower than pretending memory is infinite, but considerably less exciting when something goes wrong.


In [19]:
# making sure the long run actually produced the chunk files I expect
historical_chunk_files = sorted(
    TEMP_DIR.glob("logs_chunk_*.parquet")
)

print(
    "Historical chunk files:",
    len(historical_chunk_files)
)

Historical chunk files: 393


In [20]:
# which months did those chunks actually produce?
months = set()

for file in historical_chunk_files:
    part = pd.read_parquet(
        file,
        columns=["activity_month"]
    )
    months.update(
        part["activity_month"].unique().tolist()
    )

months = sorted(months)

print(months)
print("Number of months:", len(months))

['2015-01', '2015-02', '2015-03', '2015-04', '2015-05', '2015-06', '2015-07', '2015-08', '2015-09', '2015-10', '2015-11', '2015-12', '2016-01', '2016-02', '2016-03', '2016-04', '2016-05', '2016-06', '2016-07', '2016-08', '2016-09', '2016-10', '2016-11', '2016-12', '2017-01', '2017-02']
Number of months: 26


In [21]:
# doing this month by month so I don't try to hold the whole history in memory
sum_cols = [
    "active_days",
    "total_secs",
    "daily_unique_songs_sum",
    "songs_25",
    "songs_50",
    "songs_75",
    "songs_985",
    "songs_100"
]

monthly_output_dir = (
    PROCESSED_DIR / "user_month_history"
)

monthly_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

for month in months:
    month_parts = []

    for file in historical_chunk_files:
        part = pd.read_parquet(file)

        part = part[
            part["activity_month"] == month
        ]

        if not part.empty:
            month_parts.append(part)

    combined = pd.concat(
        month_parts,
        ignore_index=True
    )

    consolidated = (
        combined
        .groupby(
            ["msno", "activity_month"],
            as_index=False,
            observed=True
        )[sum_cols]
        .sum()
    )

    output_file = (
        monthly_output_dir /
        f"user_month_{month}.parquet"
    )

    consolidated.to_parquet(
        output_file,
        index=False
    )

    print(
        f"{month}: "
        f"{len(combined):,} partial rows -> "
        f"{len(consolidated):,} user-month rows"
    )

    del month_parts, combined, consolidated

2015-01: 12,840,005 partial rows -> 937,789 user-month rows
2015-02: 11,406,540 partial rows -> 933,040 user-month rows
2015-03: 13,173,132 partial rows -> 944,739 user-month rows
2015-04: 13,086,416 partial rows -> 939,930 user-month rows
2015-05: 13,415,932 partial rows -> 924,216 user-month rows
2015-06: 13,051,313 partial rows -> 916,862 user-month rows
2015-07: 13,125,684 partial rows -> 871,491 user-month rows
2015-08: 13,365,946 partial rows -> 920,129 user-month rows
2015-09: 13,496,961 partial rows -> 903,194 user-month rows
2015-10: 14,338,597 partial rows -> 1,012,953 user-month rows
2015-11: 14,305,843 partial rows -> 1,041,975 user-month rows
2015-12: 14,974,368 partial rows -> 1,039,271 user-month rows
2016-01: 15,049,365 partial rows -> 1,076,712 user-month rows
2016-02: 13,873,872 partial rows -> 1,041,248 user-month rows
2016-03: 15,926,525 partial rows -> 1,048,941 user-month rows
2016-04: 15,810,495 partial rows -> 1,042,406 user-month rows
2016-05: 16,447,939 partia

## 9. Add March and do one last check

The historical file ends in February 2017, so I'm saving the already-consolidated March data in the same monthly format.

That gives me a continuous 27-month activity history from January 2015 through March 2017. Before moving on, I'm checking that every monthly file has unique user-month rows and plausible active-day counts.


In [22]:
# March came from v2, but I want it stored exactly like the historical months
march_output = (
    monthly_output_dir /
    "user_month_2017-03.parquet"
)

march_monthly.to_parquet(
    march_output,
    index=False
)

print(
    f"2017-03: {len(march_monthly):,} user-month rows saved"
)

2017-03: 1,103,894 user-month rows saved


In [23]:
# final file-level check: Jan 2015 through Mar 2017 with no missing month
final_month_files = sorted(
    monthly_output_dir.glob("user_month_*.parquet")
)

print("Monthly files:", len(final_month_files))
print("First:", final_month_files[0].name)
print("Last:", final_month_files[-1].name)

Monthly files: 27
First: user_month_2015-01.parquet
Last: user_month_2017-03.parquet


In [24]:
# one last sanity check before the cohort notebook
validation = []

for file in final_month_files:
    df_month = pd.read_parquet(
        file,
        columns=["msno", "activity_month", "active_days"]
    )

    validation.append({
        "month": df_month["activity_month"].iloc[0],
        "user_months": len(df_month),
        "unique_users": df_month["msno"].nunique(),
        "duplicate_user_months": df_month.duplicated(
            ["msno", "activity_month"]
        ).sum(),
        "min_active_days": df_month["active_days"].min(),
        "max_active_days": df_month["active_days"].max()
    })

validation_df = pd.DataFrame(validation)

validation_df

,month,user_months,unique_users,duplicate_user_months,min_active_days,max_active_days
0,2015-01,937789,937789,0,1,31
1,2015-02,933040,933040,0,1,28
2,2015-03,944739,944739,0,1,31
3,2015-04,939930,939930,0,1,30
4,2015-05,924216,924216,0,1,31
5,2015-06,916862,916862,0,1,30
6,2015-07,871491,871491,0,1,31
7,2015-08,920129,920129,0,1,31
8,2015-09,903194,903194,0,1,30
9,2015-10,1012953,1012953,0,1,31
